# 3D Model Generator - TRELLIS + Groq LLM

Google Colab uzerinde calisan versiyon. Gradio arayuzu kullanilmaktadir.

---

## Baslamadan Once

1. **GPU Etkinlestir:** `Calisma Zamani > Calisma zamani turunu degistir > T4 GPU`
2. Hucreleri **sirasiyla** calistirin
3. Adim 2 bittikten sonra **runtime restart** yapin

| Bilesen | Teknoloji |
|---|---|
| 3D Donusturme | TRELLIS (microsoft/TRELLIS-image-large) |
| LLM | Groq API (llama-3.3-70b) |
| Arayuz | Gradio |
| GPU | NVIDIA T4 / A100 (Colab) |


## Adim 1 - GPU Kontrolu
> GPU'nun aktif oldugunu dogrulayin!


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadi! Calisma Zamani > T4 GPU secin')

print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'CUDA  : {torch.version.cuda}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Torch : {torch.__version__}')


## Adim 2 - Bagimlilik Kurulumu

> 5-10 dakika surebilir.  
> Kurulum bittikten sonra **runtime'i yeniden baslatin.**


In [ ]:
import subprocess, sys, torch

cuda_ver = torch.version.cuda
parts    = cuda_ver.split('.')
cuda_tag = f'cu{parts[0]}{parts[1]}'
known    = {'cu124', 'cu121', 'cu118', 'cu117', 'cu116'}
if cuda_tag not in known:
    cuda_tag = 'cu121'

spconv_pkg = f'spconv-{cuda_tag}'
print(f'CUDA {cuda_ver} -> {spconv_pkg} kurulacak')

packages = [
    spconv_pkg,
    'easydict', 'imageio', 'imageio-ffmpeg', 'plyfile',
    'rembg', 'onnxruntime-gpu', 'xatlas', 'trimesh',
    'pymeshfix', 'igraph',
    'transformers>=4.35', 'diffusers', 'accelerate',
    'groq', 'gradio>=4.0', 'plotly', 'jaxtyping', 'einops',
]

print('Paketler kuruluyor...')
for pkg in packages:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f"  {'OK' if r.returncode == 0 else 'HATA'} {pkg}")

# utils3d GitHub'dan kurulmali
print('\nutils3d GitHub dan kuruluyor...')
r = subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8',
    '-q'
], capture_output=True, text=True)
print(f"  {'OK' if r.returncode == 0 else 'HATA'} utils3d (GitHub)")

print('\n' + '='*50)
print('Kurulum tamamlandi!')
print('SIMDI: Calisma Zamani > Calisma Zamanini Yeniden Baslat')
print('Sonra Adim 3 ile devam edin.')


---
## Runtime yeniden baslatildiktan sonra buradan devam edin
---


## Adim 3 - Proje Kurulumu


In [ ]:
import os, sys

os.environ['SPCONV_ALGO']  = 'native'
os.environ['ATTN_BACKEND'] = 'sdpa'

PROJECT_DIR = '/content/3D-Model-Generator'
GITHUB_REPO = 'https://github.com/HalilIbrahimISIK/3D-Model-Generator.git'

if not os.path.exists(PROJECT_DIR):
    print('GitHub dan klonlaniyor...')
    os.system(f'git clone {GITHUB_REPO} {PROJECT_DIR}')
    print('Proje klonlandi')
    # FlexiCubes submodule'u indir
    print('Git submodules (flexicubes) indiriliyor...')
    os.system(f'cd {PROJECT_DIR}/trellis_lib && git submodule update --init --recursive')
else:
    print('Proje zaten mevcut')

for path in [PROJECT_DIR, f'{PROJECT_DIR}/trellis_lib']:
    if path not in sys.path:
        sys.path.insert(0, path)

os.chdir(PROJECT_DIR)
print(f'Calisma dizini: {os.getcwd()}')

import torch
assert torch.cuda.is_available(), 'GPU bulunamadi!'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'CUDA : {torch.version.cuda}')


## Adim 4 - HuggingFace Token (ZORUNLU)

> TRELLIS gated modeldir - erisim icin HuggingFace hesabi ve token gerekli!

**Adimlar:**
1. https://huggingface.co/microsoft/TRELLIS-image-large -> 'Agree and access repository'
2. https://huggingface.co/settings/tokens -> 'New token' -> Read -> Token kopyala
3. Asagidaki hucreye token yapistir veya Colab Secrets kullan


In [ ]:
from huggingface_hub import login

# Token giris (direkt yaz veya Secrets kullan)
HF_TOKEN = ''  # 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# Colab Secrets kullaniyorsaniz (Sol panel > Secrets > HF_TOKEN):
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('HF_TOKEN Secrets tan okundu')
except Exception:
    pass

if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    raise ValueError(
        'HF_TOKEN girilmedi!\\n'
        '1. https://huggingface.co/microsoft/TRELLIS-image-large -> Agree tikla\\n'
        '2. https://huggingface.co/settings/tokens -> token al\\n'
        '3. HF_TOKEN = \'hf_xxx...\' yapistir VEYA Secrets kullan'
    )

login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace girisi basarili!')
print(f'Token: {HF_TOKEN[:8]}...{HF_TOKEN[-8:]}')


## Adim 5 - TRELLIS Modelini Yukle

> Ilk calistirmada model HuggingFace'den indirilir (~2 GB).  
> Sonraki calistirmalarda cache'den okunur.


In [ ]:
from trellis.pipelines import TrellisImageTo3DPipeline

print('TRELLIS modeli yukleniyor...')
print('Ilk seferde ~2 GB indirilir, lutfen bekleyin')

pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()

print('TRELLIS modeli hazir ve GPU da!')


## Adim 6 - Groq API Anahtari (Istege Bagli)

LLM chat icin gerekli. STL donusturme icin **gerekli degil.**  
Ucretsiz anahtar: https://console.groq.com/


In [ ]:
import os

# Yontem A: Direkt yaz
GROQ_API_KEY = ''  # 'gsk_xxxxxxxxxxxxxxxx'

# Yontem B: Colab Secrets (Sol panel > Secrets > GROQ_API_KEY)
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print('Groq API anahtari ayarlandi')
else:
    print('Groq API anahtari girilmedi -- LLM chat devre disi')
    print('STL donusturme yine de calisacak')


## Adim 7 - Gradio Uygulamasini Baslat

> Hucreyi calistirdiktan sonra **public link** olusturulacak.  
> Link **72 saat** gecerlidir.


In [ ]:
import gradio as gr
import trimesh, numpy as np, os, datetime, torch
from PIL import Image
import plotly.graph_objects as go


def _path(f):
    if f is None: return None
    if isinstance(f, str): return f
    if isinstance(f, dict): return f.get('name') or f.get('path', '')
    if hasattr(f, 'name'): return f.name
    return str(f)


def convert_to_stl(files, steps, cfg_strength, progress=gr.Progress()):
    if not files:
        return None, None, 'Gorsel yuklenmedi.'
    paths = [_path(f) for f in files if _path(f)]
    n = len(paths)
    try:
        progress(0.1, desc='Gorseller yukleniyor...')
        images = [Image.open(p).convert('RGB') for p in paths]
        progress(0.3, desc=f'{n} gorsel ile 3D yeniden yapilandirma...')
        params = {
            'seed': 42, 'preprocess_image': True,
            'sparse_structure_sampler_params': {'steps': int(steps), 'cfg_strength': float(cfg_strength)},
            'slat_sampler_params': {'steps': int(steps), 'cfg_strength': 3.0},
        }
        with torch.no_grad():
            outputs = pipeline.run(images[0], **params) if n == 1 else pipeline.run_multi_image(images, **params)
        progress(0.8, desc='Mesh cikariliyor...')
        mesh_result = outputs['mesh'][0]
        vertices = mesh_result.vertices.cpu().float().numpy()
        faces    = mesh_result.faces.cpu().numpy()
        tri_mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
        progress(0.9, desc='STL kaydediliyor...')
        ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        stl_path = f'/content/model_{"multi" + str(n) if n > 1 else "single"}_{ts}.stl'
        tri_mesh.export(stl_path)
        fig = go.Figure(data=[go.Mesh3d(
            x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
            i=faces[:,0], j=faces[:,1], k=faces[:,2],
            opacity=0.75, colorscale='Viridis', intensity=vertices[:,2], showscale=False,
        )])
        fig.update_layout(
            scene=dict(aspectmode='data', bgcolor='#0f0f1a'),
            paper_bgcolor='#0f0f1a', font_color='white',
            title=dict(text=f'3D Model - {n} gorsel', font_color='#c8b8ff'),
            height=500, margin=dict(l=0, r=0, t=40, b=0),
        )
        size_kb  = os.path.getsize(stl_path) / 1024
        size_str = f'{size_kb:.0f} KB' if size_kb < 1024 else f'{size_kb/1024:.2f} MB'
        status = (
            f'Donusturme tamamlandi!\n'
            f'Dosya : {os.path.basename(stl_path)}\n'
            f'Boyut : {size_str}\n'
            f'Mesh  : {len(vertices):,} vertex - {len(faces):,} yuz\n'
            f'Mod   : {n} gorsel - {"cok aci" if n > 1 else "tek gorsel"}'
        )
        progress(1.0, desc='Tamamlandi!')
        return stl_path, fig, status
    except Exception as e:
        return None, None, f'Hata: {str(e)}'


def llm_chat(message, history, api_key, image_file):
    if not message.strip():
        return history, ''
    if not api_key.strip():
        history = history or []
        history.append({'role': 'assistant', 'content': 'Groq API anahtari girilmedi.'})
        return history, ''
    try:
        from groq import Groq
        import base64
        client   = Groq(api_key=api_key.strip())
        model    = 'llama-3.3-70b-versatile'
        content  = message
        img_path = _path(image_file)
        if img_path and os.path.exists(img_path):
            ext  = os.path.splitext(img_path)[1].lower()
            mime = {'jpg': 'image/jpeg', 'jpeg': 'image/jpeg', 'png': 'image/png'}.get(ext[1:], 'image/jpeg')
            with open(img_path, 'rb') as f:
                b64 = base64.b64encode(f.read()).decode()
            content = [{'type': 'text', 'text': message},
                       {'type': 'image_url', 'image_url': {'url': f'data:{mime};base64,{b64}'}}]
            model = 'llama-3.2-11b-vision-preview'
        msgs = [{'role': 'system', 'content': 'Sen 3D model asistanisin. Turkce konusuyorsun.'}]
        history = history or []
        for h in history:
            msgs.append({'role': h['role'], 'content': h['content']})
        msgs.append({'role': 'user', 'content': content})
        resp   = client.chat.completions.create(model=model, messages=msgs, max_tokens=1024)
        answer = resp.choices[0].message.content
        history.append({'role': 'user', 'content': message})
        history.append({'role': 'assistant', 'content': answer})
        return history, ''
    except Exception as e:
        history = history or []
        history.append({'role': 'assistant', 'content': f'Hata: {str(e)}'})
        return history, ''


with gr.Blocks(title='3D Model Generator', theme=gr.themes.Soft(primary_hue='violet')) as demo:
    gr.Markdown('# 3D Model Generator - TRELLIS + Groq')
    gr.Markdown('Tek veya birden fazla acidan gorsel yukleyerek STL 3D modeli olusturun.')

    with gr.Tabs():
        with gr.Tab('Gorsel -> STL'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Gorsel Yukle')
                    gr.Markdown('Farkli acilardan birden fazla gorsel daha iyi 3D sonuc verir.')
                    images_input = gr.File(label='Gorsel(ler)', file_count='multiple',
                                          file_types=['.png', '.jpg', '.jpeg', '.webp', '.bmp'])
                    steps_slider = gr.Slider(8, 30, 12, step=2, label='Diffusion Adimlari')
                    cfg_slider   = gr.Slider(1.0, 15.0, 7.5, step=0.5, label='CFG Gucu')
                    convert_btn  = gr.Button('STL e Donustur', variant='primary', size='lg')
                with gr.Column(scale=1):
                    gr.Markdown('### Cikti')
                    status_box = gr.Textbox(label='Durum', lines=5, interactive=False)
                    stl_output = gr.File(label='STL Dosyasi Indir')
            preview_plot = gr.Plot(label='3D Onizleme')
            convert_btn.click(fn=convert_to_stl,
                              inputs=[images_input, steps_slider, cfg_slider],
                              outputs=[stl_output, preview_plot, status_box])
            gr.Markdown('### Ornek Gorseller')
            gr.Examples(
                examples=[
                    [['trellis_lib/assets/example_multi_image/rabbit_1.png',
                      'trellis_lib/assets/example_multi_image/rabbit_2.png',
                      'trellis_lib/assets/example_multi_image/rabbit_3.png'], 12, 7.5],
                    [['trellis_lib/assets/example_multi_image/mushroom_1.png',
                      'trellis_lib/assets/example_multi_image/mushroom_2.png',
                      'trellis_lib/assets/example_multi_image/mushroom_3.png'], 12, 7.5],
                    [['trellis_lib/assets/example_image/typical_creature_dragon.png'], 12, 7.5],
                ],
                inputs=[images_input, steps_slider, cfg_slider],
            )

        with gr.Tab('LLM Chat (Groq)'):
            gr.Markdown('AI asistaniyla sohbet edin. Gorsel yukleyerek analiz yaptirilabilir.')
            with gr.Row():
                api_key_input = gr.Textbox(label='Groq API Anahtari', type='password',
                                           placeholder='gsk_xx...', scale=3,
                                           value=os.environ.get('GROQ_API_KEY', ''))
                chat_image    = gr.File(label='Gorsel (istege bagli)',
                                       file_types=['.png', '.jpg', '.jpeg', '.webp'], scale=1)
            chatbot   = gr.Chatbot(label='Sohbet', height=400, type='messages')
            with gr.Row():
                msg_input = gr.Textbox(label='Mesaj', scale=4)
                send_btn  = gr.Button('Gonder', variant='primary', scale=1)
            clear_btn = gr.Button('Temizle', size='sm')
            send_btn.click(fn=llm_chat,
                           inputs=[msg_input, chatbot, api_key_input, chat_image],
                           outputs=[chatbot, msg_input])
            msg_input.submit(fn=llm_chat,
                             inputs=[msg_input, chatbot, api_key_input, chat_image],
                             outputs=[chatbot, msg_input])
            clear_btn.click(fn=lambda: ([], ''), outputs=[chatbot, msg_input])

        with gr.Tab('Bilgi'):
            vram_gb = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
            gr.Markdown(
                '## Sistem Bilgisi\n'
                f'- GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok"}\n'
                f'- VRAM : {vram_gb:.1f} GB\n'
                f'- CUDA : {torch.version.cuda}\n'
                f'- Torch: {torch.__version__}\n'
                '\n## Cok Gorsel Modu\n'
                '- Ayni nesnenin 2-4 farkli acidan fotografini yukleyin\n'
                '- Sonuc cok daha gercekci ve detayli olur\n'
                '- Onerilir: on, yan, ust acilari\n'
            )

print('Gradio uygulamasi baslatiliyor...')
demo.launch(share=True, debug=False)


---
## Sorun Giderme

### spconv import hatasi
```python
import torch; print(torch.version.cuda)
!pip install spconv-cu121  # CUDA 12.1
!pip install spconv-cu118  # CUDA 11.8
```

### GPU bellek hatasi (OOM)
```python
import torch, gc
gc.collect(); torch.cuda.empty_cache()
# Adim sayisini 8 e dusurun
```

### TRELLIS import hatasi
```python
import os
os.environ['ATTN_BACKEND'] = 'sdpa'
os.environ['SPCONV_ALGO']  = 'native'
# Sonra Adim 3 ten devam edin
```
